# 🎬 EducationVdo - Colab Runner

Run **EducationVdo** with full GPU acceleration on Google Colab.
Supports face swapping, face enhancement, expression restoration, and **image-to-video AI animation (LivePortrait)**.

In [ ]:
# ── Check GPU Availability ───────────────────────────────
!nvidia-smi

In [ ]:
# ── Step 1: Clone Repository ─────────────────────────────
from getpass import getpass
import os

print("If your repository is private, enter your GitHub Personal Access Token.")
print("Generate one at: https://github.com/settings/tokens")
print("Leave empty if the repository is public.")
print()

token = getpass("GitHub Token (optional): ")

repo_url = "https://github.com/hasnatvip/educatonvdo.git"
if token.strip():
    repo_url = repo_url.replace("https://", f"https://{token}@")

# Remove existing directory if it exists
!rm -rf educatonvdo

# Clone the repository
!git clone {repo_url}

# Change to the project directory
%cd educatonvdo

In [ ]:
# ── Step 2: Install Dependencies & Setup CUDA 12 ─────────
print("Installing dependencies and CUDA 12 runtime libraries...")

# Ensure clean onnxruntime environment
!pip uninstall -q -y onnxruntime onnxruntime-directml onnxruntime-openvino 2>/dev/null || true

# Install project requirements
!pip install -q -r requirements.txt

# Install NVIDIA CUDA 12 & cuDNN libraries (fixes libcublasLt.so.12 / libcublas.so.12)
!pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12

# Configure library search paths for dynamic linker
import os, site
nvidia_lib_dirs = []
for p in site.getsitepackages():
    nvidia_dir = os.path.join(p, "nvidia")
    if os.path.isdir(nvidia_dir):
        for sub in os.listdir(nvidia_dir):
            lib_path = os.path.join(nvidia_dir, sub, "lib")
            if os.path.isdir(lib_path):
                nvidia_lib_dirs.append(lib_path)

cuda_system_dirs = ["/usr/local/cuda/lib64", "/usr/local/cuda-12/lib64"]
all_lib_dirs = [d for d in nvidia_lib_dirs + cuda_system_dirs if os.path.isdir(d)]
existing_ld = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = ":".join(all_lib_dirs + ([existing_ld] if existing_ld else []))

# Verify installation & CUDA Provider
import onnxruntime
providers = onnxruntime.get_available_providers()
print()
print(f"✓ ONNX Runtime: {onnxruntime.__version__}")
print(f"✓ Available Execution Providers: {providers}")

if "CUDAExecutionProvider" in providers:
    print("🚀 CUDA Acceleration is READY!")
else:
    print("⚠️ Notice: Running with CPU (CUDA provider not initialized)")

print()
print("✓ Setup completed successfully!")

In [ ]:
# ── Step 3: Launch EducationVdo UI ───────────────────────
import onnxruntime

provider = "cuda" if "CUDAExecutionProvider" in onnxruntime.get_available_providers() else "cpu"
print(f"Launching EducationVdo with {provider} execution provider...")
print("Look for the public Gradio link below (e.g. https://...gradio.live)")
print()

!python educationvdo.py run --execution-providers {provider}